# Neurotypical vs Pathological — FeTa protocol

All comparisons are restricted to **FeTa-protocol subjects** so neurotypical and pathological are on the same footing.

- Neurotypical FeTa: **n=9** (from the mixed neurotypical split)
- Pathological FeTa: **n=11**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

STRUCT_NAMES = {
    1: 'CSF', 2: 'Cortical GM', 3: 'White Matter',
    4: 'Ventricles', 5: 'Cerebellum', 6: 'Deep GM', 7: 'Brainstem',
}
FETA_STRUCTS = list(STRUCT_NAMES.values())
dice_cols = [f'dice_ch_{i:02d}' for i in range(1, 8)]
DICE_RENAME = {f'dice_ch_{i:02d}': n for i, n in STRUCT_NAMES.items()}

# Okabe-Ito colorblind-safe palette
MODEL_PALETTE = {
    'CoNeMOS drawEM9-cond': '#E69F00',   # orange
    'CoNeMOS FeTa-cond': '#56B4E9',   # sky blue
    'FeTa specialist':   '#009E73',   # bluish green
    'dHCP specialist':   '#D55E00',   # vermilion
}
MODELS = list(MODEL_PALETTE.keys())

In [ ]:
NEURO_PATHS = {
    'CoNeMOS drawEM9-cond': '../eval_results/conemos_dhcp_cond_neurotypical/results.csv',
    'CoNeMOS FeTa-cond': '../eval_results/conemos_feta_cond_neurotypical/results.csv',
    'FeTa specialist':   '../eval_results/feta_specialist_neurotypical/results.csv',
    'dHCP specialist':   '../eval_results/dhcp_specialist_neurotypical/results.csv',
}
PATHO_PATHS = {
    'CoNeMOS drawEM9-cond': '../eval_results/conemos_dhcp_cond_feta_pathological/results.csv',
    'CoNeMOS FeTa-cond': '../eval_results/conemos_feta_cond_feta_pathological/results.csv',
    'FeTa specialist':   '../eval_results/feta_specialist_feta_pathological/results.csv',
    'dHCP specialist':   '../eval_results/dhcp_specialist_feta_pathological/results.csv',
}

# restrict neurotypical to FeTa protocol
dfs_n = {n: pd.read_csv(p).query("protocol == 'feta'") for n, p in NEURO_PATHS.items()}
dfs_p = {n: pd.read_csv(p) for n, p in PATHO_PATHS.items()}

print('N subjects — neurotypical FeTa:', len(next(iter(dfs_n.values()))))
print('N subjects — pathological FeTa:', len(next(iter(dfs_p.values()))))

## 1 · Overall mean Dice: neurotypical vs pathological

In [ ]:
summary_rows = []
for name in MODELS:
    n_mean = dfs_n[name]['mean_dice'].mean()
    n_std  = dfs_n[name]['mean_dice'].std()
    p_mean = dfs_p[name]['mean_dice'].mean()
    p_std  = dfs_p[name]['mean_dice'].std()
    summary_rows.append({
        'Model': name,
        'Neurotypical': f'{n_mean:.3f} ± {n_std:.3f}',
        'Pathological': f'{p_mean:.3f} ± {p_std:.3f}',
        'Drop': f'{p_mean - n_mean:+.3f}',
        'Retained (%)': f'{100 * p_mean / n_mean:.1f}%',
    })
display(pd.DataFrame(summary_rows).set_index('Model'))

In [ ]:
rows = []
for name in MODELS:
    n_mean = dfs_n[name]['mean_dice'].mean()
    n_std  = dfs_n[name]['mean_dice'].std()
    p_mean = dfs_p[name]['mean_dice'].mean()
    p_std  = dfs_p[name]['mean_dice'].std()
    rows.append({
        'Model':           name,
        'Neuro (FeTa)':    f'{n_mean:.3f} ± {n_std:.3f}',
        'Patho (FeTa)':    f'{p_mean:.3f} ± {p_std:.3f}',
        'Drop':            f'{p_mean - n_mean:+.3f}',
        'Retained (%)':    f'{100 * p_mean / n_mean:.1f}%',
    })

display(
    pd.DataFrame(rows)
      .set_index('Model')
      .style.set_caption(
          'Mean Dice — FeTa neurotypical (n=9) vs pathological (n=11). '
          'Retained = pathological / neurotypical.'
      )
)

## 2 · Side-by-side bar chart: neurotypical vs pathological

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(MODELS))
w = 0.35

n_means = [dfs_n[m]['mean_dice'].mean() for m in MODELS]
n_stds  = [dfs_n[m]['mean_dice'].std()  for m in MODELS]
p_means = [dfs_p[m]['mean_dice'].mean() for m in MODELS]
p_stds  = [dfs_p[m]['mean_dice'].std()  for m in MODELS]
colors  = [MODEL_PALETTE[m] for m in MODELS]

bars_n = ax.bar(x - w/2, n_means, w, yerr=n_stds, capsize=4,
                color=colors, alpha=0.5, label='Neurotypical', hatch='')
bars_p = ax.bar(x + w/2, p_means, w, yerr=p_stds, capsize=4,
                color=colors, alpha=0.95, label='Pathological')

# draw drop arrows
for i, (nm, pm) in enumerate(zip(n_means, p_means)):
    ax.annotate('', xy=(i + w/2, pm + 0.01), xytext=(i - w/2, nm - 0.01),
                arrowprops=dict(arrowstyle='->', color='grey', lw=1.2))
    ax.text(i + 0.02, (nm + pm) / 2, f'{pm - nm:+.2f}',
            ha='left', va='center', fontsize=8.5, color='grey')

ax.set_xticks(x)
ax.set_xticklabels(MODELS, rotation=12, ha='right')
ax.set_ylabel('Mean Dice')
ax.set_ylim(0, 1.05)
ax.set_title('Mean Dice — FeTa neurotypical (n=9) vs pathological (n=11)')

neuro_patch = mpatches.Patch(facecolor='grey', alpha=0.4, label='Neurotypical')
patho_patch = mpatches.Patch(facecolor='grey', alpha=0.95, label='Pathological')
ax.legend(handles=[neuro_patch, patho_patch], loc='lower left')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3 · Per-subject strip plot — both splits together

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 5), sharey=True)

for ax, name in zip(axes, MODELS):
    color = MODEL_PALETTE[name]
    n_vals = dfs_n[name]['mean_dice'].values
    p_vals = dfs_p[name]['mean_dice'].values

    ax.scatter(np.zeros(len(n_vals)), n_vals,
               color=color, s=60, alpha=0.5, marker='o', label='Neurotypical')
    ax.scatter(np.ones(len(p_vals)),  p_vals,
               color=color, s=60, alpha=0.95, marker='D', label='Pathological')

    # mean lines
    ax.hlines(np.mean(n_vals), -0.3, 0.3, color=color, linewidth=2, alpha=0.5)
    ax.hlines(np.mean(p_vals),  0.7, 1.3, color=color, linewidth=2)

    ax.set_xlim(-0.5, 1.5)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Neuro\n(n=9)', 'Patho\n(n=11)'])
    ax.set_title(name, fontsize=10)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, axis='y', alpha=0.3)

axes[0].set_ylabel('Mean Dice')
plt.suptitle('FeTa: neurotypical vs pathological per model', fontsize=13)
plt.tight_layout()
plt.show()

## 4 · Drop from neurotypical → pathological per structure

Which structures are most affected by pathology, and does the conditioning gap widen?

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 5))

for ax, name in zip(axes, MODELS):
    drops = [dfs_p[name][dc].mean() - dfs_n[name][dc].mean() for dc in dice_cols]
    bar_colors = ['#2196F3' if v >= 0 else '#F44336' for v in drops]
    ax.barh(range(len(FETA_STRUCTS)), drops, color=bar_colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(range(len(FETA_STRUCTS)))
    ax.set_yticklabels(FETA_STRUCTS)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Δ Dice (patho − neuro)')
    overall = dfs_p[name]['mean_dice'].mean() - dfs_n[name]['mean_dice'].mean()
    ax.text(0.97, 0.03, f'Overall: {overall:+.3f}',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    ax.grid(True, axis='x', alpha=0.3)

plt.suptitle('Per-structure drop: pathological − neurotypical', fontsize=13)
plt.tight_layout()
plt.show()

## 5 · Conditioning gap: does it widen under pathology?

In [ ]:
# Gap = CoNeMOS FeTa-cond minus CoNeMOS drawEM9-cond, per structure, per split
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, (label, dfs) in zip(axes, [('Neurotypical FeTa (n=9)', dfs_n),
                                     ('Pathological FeTa (n=11)', dfs_p)]):
    feta_means = [dfs['CoNeMOS FeTa-cond'][dc].mean() for dc in dice_cols]
    dhcp_means = [dfs['CoNeMOS drawEM9-cond'][dc].mean() for dc in dice_cols]
    gap = [f - d for f, d in zip(feta_means, dhcp_means)]

    x = np.arange(len(FETA_STRUCTS))
    ax.bar(x, feta_means, 0.35, label='FeTa-cond',
           color=MODEL_PALETTE['CoNeMOS FeTa-cond'], alpha=0.85)
    ax.bar(x + 0.37, dhcp_means, 0.35, label='drawEM9-cond',
           color=MODEL_PALETTE['CoNeMOS drawEM9-cond'], alpha=0.85)

    for i, g in enumerate(gap):
        ax.text(i + 0.185, max(feta_means[i], dhcp_means[i]) + 0.02,
                f'{g:+.2f}', ha='center', va='bottom', fontsize=8, color='black')

    overall_gap = (dfs['CoNeMOS FeTa-cond']['mean_dice'].mean()
                   - dfs['CoNeMOS drawEM9-cond']['mean_dice'].mean())
    ax.set_xticks(x + 0.185)
    ax.set_xticklabels(FETA_STRUCTS, rotation=25, ha='right')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Mean Dice')
    ax.set_title(f'{label}\n(overall gap FeTa-cond − drawEM9-cond = {overall_gap:+.3f})')
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('CoNeMOS FeTa-cond vs drawEM9-cond: does the conditioning gap widen under pathology?',
             fontsize=12)
plt.tight_layout()
plt.show()

## 6 · Summary: robustness to pathology per model

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for name in MODELS:
    n_mean = dfs_n[name]['mean_dice'].mean()
    p_mean = dfs_p[name]['mean_dice'].mean()
    color  = MODEL_PALETTE[name]
    ax.scatter([n_mean], [p_mean], color=color, s=120, zorder=4, label=name)
    ax.annotate(name, (n_mean, p_mean),
                textcoords='offset points', xytext=(6, 4), fontsize=8.5)

# y = x line (perfect robustness)
lims = [0.3, 1.0]
ax.plot(lims, lims, 'k--', linewidth=1, alpha=0.4, label='y = x (no drop)')
ax.set_xlim(*lims)
ax.set_ylim(*lims)
ax.set_xlabel('Mean Dice — FeTa neurotypical')
ax.set_ylabel('Mean Dice — FeTa pathological')
ax.set_title('Neurotypical vs pathological performance\n(closer to diagonal = more robust)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()